# NuNo vs RPT — Colab G4 paper-config comparison
Cấu hình Qwen bám Table 5 của paper: **14B → 1.5B**, global batch 64, LR 5e-6, cosine + 10% warmup, length 1024, 2 epochs, λ=0.2, K=128, d′=256, 4 layers trong [0.20, 0.85].

> Colab G4 dùng RTX PRO 6000 Blackwell 96 GB, nên notebook chạy teacher 14B FP16 nguyên bản (`TEACHER_4BIT=False`). Nhánh 4-bit chỉ tự bật nếu notebook bị chuyển sang GPU dưới 35 GB.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'Hãy chọn Runtime > Change runtime type > GPU'
gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(gpu_name, round(gpu_gb, 1), 'GB')

In [ ]:
%pip install -q transformers==4.57.3 peft==0.18.1 datasets deepspeed bitsandbytes accelerate rouge-score numerize rich wandb celery nltk

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, shlex, subprocess, time, zipfile

def run_live(cmd, env=None):
    """Chạy command và stream stdout + stderr trực tiếp vào output cell."""
    live_env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    if env:
        live_env.update(env)
    print('$', shlex.join([str(x) for x in cmd]), flush=True)
    started = time.time()
    process = subprocess.Popen(
        [str(x) for x in cmd], env=live_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    elapsed = time.time() - started
    print(f'\n[finished] exit={return_code} time={elapsed/60:.1f} min', flush=True)
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
    return return_code
REPO = Path('/content/nuno-kd')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/chiiipk/nuno-kd.git', str(REPO)], check=True)
os.chdir(REPO)
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
subprocess.run(['python3', '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)

## Data
Upload file ZIP vào Colab (`/content`) hoặc bất kỳ thư mục nào trong Google Drive. Cell sẽ tự tìm archive có `generated_train.jsonl`, giải nén và ưu tiên dữ liệu Qwen.

In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/NuNo')
EXTRACT_ROOT = Path('/content/nuno_data')

# Tìm JSONL đã giải nén sẵn trong Drive hoặc /content.
search_roots = [Path('/content'), Path('/content/drive/MyDrive')]
existing_jsonl = []
for root in search_roots:
    if root.exists():
        existing_jsonl.extend(root.rglob('generated_train.jsonl'))

# Tìm mọi ZIP, sau đó chỉ nhận archive thực sự chứa generated_train.jsonl.
all_zips = []
for root in search_roots:
    if root.exists():
        all_zips.extend(root.rglob('*.zip'))
all_zips = sorted(set(all_zips), key=lambda p: (
    0 if p.name.lower() in {'data.zip', 'nuno.zip'} else 1, str(p)
))
zip_files = []
for archive in all_zips:
    try:
        with zipfile.ZipFile(archive) as zf:
            if any(Path(name).name == 'generated_train.jsonl' for name in zf.namelist()):
                zip_files.append(archive)
    except (zipfile.BadZipFile, OSError):
        pass
print('ZIP chứa dataset:', [str(p) for p in zip_files])

extracted_jsonl = list(EXTRACT_ROOT.rglob('generated_train.jsonl')) if EXTRACT_ROOT.exists() else []
if not extracted_jsonl and zip_files:
    archive = zip_files[0]
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    print('Đang giải nén:', archive, '->', EXTRACT_ROOT)
    with zipfile.ZipFile(archive, 'r') as zf:
        zf.extractall(EXTRACT_ROOT)
    print('Giải nén xong')

candidates = list(existing_jsonl)
if EXTRACT_ROOT.exists():
    candidates.extend(EXTRACT_ROOT.rglob('generated_train.jsonl'))
candidates = sorted(set(candidates))
qwen = [p for p in candidates if 'Qwen' in str(p) or 'qwen' in str(p)]
RAW_JSONL = (qwen or candidates or [None])[0]
assert RAW_JSONL and RAW_JSONL.exists(), (
    'Không tìm thấy generated_train.jsonl hoặc ZIP chứa file này trong /content và MyDrive'
)
with RAW_JSONL.open() as f:
    first = json.loads(next(f))
assert {'prompt', 'generated_text'} <= first.keys()
print('Data:', RAW_JSONL, '| MB:', round(RAW_JSONL.stat().st_size/2**20, 1))

In [ ]:
# ===== Paper configuration (Qwen column, Table 5) =====
STUDENT = 'Qwen/Qwen2.5-1.5B-Instruct'
TEACHER = 'Qwen/Qwen2.5-14B-Instruct'
GLOBAL_BATCH = 64
MICRO_BATCH = 1
GRAD_ACC = GLOBAL_BATCH // MICRO_BATCH  # Colab: one GPU
LR = '5e-6'
EPOCHS = 2
MAX_LENGTH = 1024
WARMUP_RATIO = '0.1'
NNM_RATIO = '0.2'
CST_LOSS_WEIGHT = '0.003'  # tuned separately because raw CST loss has a different scale
K_CENTROIDS = '128'
D_PRIME = '256'
CENTROID_BATCHES = '500'
N_LAYERS = '4'
TEACHER_4BIT = gpu_gb < 35  # False trên G4 RTX PRO 6000 96 GB
SEED = 10
print('Teacher 4-bit:', TEACHER_4BIT, '| effective batch:', MICRO_BATCH * GRAD_ACC)

In [ ]:
PROCESSED_ROOT = Path('/content/processed_data')
DATA_DIR = PROCESSED_ROOT / TEACHER
if not (DATA_DIR / 'train_0.idx').exists():
    run_live([
        'python3', 'tools/process_data_ultraInteract.py', '--data-dir', str(RAW_JSONL),
        '--processed-data-dir', str(PROCESSED_ROOT), '--model-path', TEACHER,
        '--data-process-workers', '2', '--max-prompt-length', '512',
        '--max-length', str(MAX_LENGTH), '--dev-num', '200', '--only-prompt',
        '--model-type', 'qwen'
    ], env={'PYTHONPATH': '.'})
assert (DATA_DIR / 'train_0.idx').exists() and (DATA_DIR / 'valid_0.idx').exists()
print('Processed:', DATA_DIR)

In [ ]:
def run_paper_config(variant: str, train_num: int = -1):
    assert variant in {'none', 'nuno', 'rpt', 'cst'}
    output = DATA_ROOT / 'results' / f'{variant}_paper_qwen14b_to_1p5b'
    cmd = [
      'torchrun', '--standalone', '--nproc_per_node=1', 'finetune.py',
      '--base-path', '.', '--model-path', STUDENT, '--teacher-model-path', TEACHER,
      '--ckpt-name', 'qwen2.5-1.5B-it', '--teacher-ckpt-name', 'qwen2.5-14B-it',
      '--teacher-model-fp16', '--n-gpu', '1', '--data-dir', str(DATA_DIR),
      '--num-workers', '2', '--train-num', str(train_num), '--dev-num', '-1',
      '--lr', LR, '--lr-min', '0', '--batch-size', str(MICRO_BATCH),
      '--eval-batch-size', '1', '--gradient-accumulation-steps', str(GRAD_ACC),
      '--gradient-checkpointing', '--warmup-ratio', WARMUP_RATIO,
      '--lr-decay-style', 'cosine', '--weight-decay', '1e-2', '--clip-grad', '1.0',
      '--epochs', str(EPOCHS), '--kd-ratio', '1.0', '--temperature', '1.0',
      '--max-length', str(MAX_LENGTH), '--max-prompt-length', '512', '--do-train',
      '--save-interval', '-1', '--eval-interval', '-1', '--log-interval', '10',
      '--mid-log-num', '-1', '--save', str(output), '--seed', str(SEED),
      '--deepspeed', '--deepspeed_config', 'configs/deepspeed/ds_config_zero2_offload.json',
      '--type', 'adaptive-sfkl', '--skew-alpha', '0.1', '--do-sample', '--top-k', '0',
      '--top-p', '1.0', '--student-gen', '--gen-num-beams', '1', '--gen-top-p', '1.0',
      '--init-threshold', '0.0', '--loss-eps', '0.1', '--capacity', '1000',
      '--replay-ratio', 'decreasing', '--mixed-alpha', '0.5', '--delta-threshold', '0.03'
    ]
    if variant == 'none':
        cmd.append('--no-nnm')
    else:
        cmd += [
          '--nnm', '--loss-variant', variant, '--nnm-ratio', NNM_RATIO,
          '--nnm-K', K_CENTROIDS, '--nnm-n-layers', N_LAYERS,
          '--nnm-d-prime', D_PRIME, '--nnm-centroid-batches', CENTROID_BATCHES,
          '--nnm-eta', '0.05', '--nnm-T-dead', '50', '--nnm-ns-iters', '5',
          '--rpt-max-tokens', '64'
        ]
    if variant == 'cst':
        cmd += [
          '--cst-loss-weight', CST_LOSS_WEIGHT, '--cst-max-tokens', '64',
          '--cst-num-layers', '4', '--cst-layer-min', '0.20',
          '--cst-layer-max', '0.85', '--cst-gamma-min', '1e-2',
          '--cst-gamma-max', '1e2', '--cst-num-gamma-samples', '2',
          '--cst-gamma-sampling', 'log_uniform', '--cst-distance', 'l2'
        ]
    if TEACHER_4BIT:
        cmd.append('--teacher-load-in-4bit')
    env = {'PYTHONPATH': '.', 'WANDB_DISABLED': 'true', 'PYTHONUNBUFFERED': '1',
           'TOKENIZERS_PARALLELISM': 'false', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}
    print('Running', variant, '->', output)
    run_live(cmd, env=env)
    return output

## Chạy
Smoke test 64 mẫu trước. Sau khi pass, restart runtime để giải phóng VRAM rồi chạy từng experiment. Không chạy các variant đồng thời. CST mặc định dùng m=64, q=2, n=4.

In [ ]:
run_paper_config('cst', train_num=64)

In [ ]:
# Full run — bỏ comment đúng một dòng mỗi runtime:
# run_paper_config('none', train_num=-1)   # output KD baseline
# run_paper_config('nuno', train_num=-1)
# run_paper_config('rpt', train_num=-1)
# run_paper_config('cst', train_num=-1)

## Benchmark evaluation giống NuNo
Chạy section này **sau khi training kết thúc** (nên restart runtime trước để giải phóng VRAM). Nó tự chọn checkpoint số lớn nhất, chạy lm-evaluation-harness bằng vLLM trên 1 GPU và lưu từng kết quả vào Google Drive. `limit` chỉ dùng để kiểm tra pipeline; kết quả báo cáo phải dùng `limit=None`.

In [ ]:
%pip install -q -U "lm_eval[vllm,math]"

# Nếu vừa training trong cùng runtime, restart runtime rồi chạy lại các cell
# mount Drive/import/run_live trước khi bắt đầu evaluation.
import gc, importlib.util, sys, torch
assert importlib.util.find_spec('lm_eval') is not None, 'lm_eval chưa được cài trong kernel hiện tại'
print('lm_eval module: OK | Python:', sys.executable)
gc.collect()
torch.cuda.empty_cache()

In [ ]:
def latest_hf_checkpoint(variant: str) -> Path:
    run_dir = DATA_ROOT / 'results' / f'{variant}_paper_qwen14b_to_1p5b'
    checkpoints = [p for p in run_dir.iterdir() if p.is_dir() and p.name.isdigit()]
    checkpoints = [p for p in checkpoints if (p / 'config.json').exists()]
    assert checkpoints, f'Không tìm thấy Hugging Face checkpoint trong {run_dir}'
    checkpoint = max(checkpoints, key=lambda p: int(p.name))
    print('Checkpoint:', checkpoint)
    return checkpoint

BENCHMARKS = [
    ('gsm8k', None, False),
    ('minerva_math', 4, False),
    ('mmlu_stem', 5, False),
    ('sciq', None, False),
    ('mbpp', 3, True),
    ('gsm_plus', None, False),
    ('mmlu_pro_math', None, False),
    ('bbh_cot_fewshot', None, False),
]

def evaluate_benchmarks(variant='cst', limit=None, tasks=BENCHMARKS):
    checkpoint = latest_hf_checkpoint(variant)
    result_root = DATA_ROOT / 'benchmark_results' / f'{variant}_step_{checkpoint.name}'
    result_root.mkdir(parents=True, exist_ok=True)
    model_args = (
        f'pretrained={checkpoint},tensor_parallel_size=1,dtype=bfloat16,'
        'gpu_memory_utilization=0.8,trust_remote_code=True'
    )
    for task, fewshot, unsafe_code in tasks:
        cmd = [
            sys.executable, '-m', 'lm_eval', '--model', 'vllm', '--model_args', model_args,
            '--tasks', task, '--batch_size', 'auto', '--log_samples',
            '--output_path', str(result_root / task),
            '--gen_kwargs', 'max_new_tokens=5120,temperature=0.0',
        ]
        if task != 'mbpp':
            cmd += ['--apply_chat_template', '--fewshot_as_multiturn']
        if fewshot is not None:
            cmd += ['--num_fewshot', str(fewshot)]
        if unsafe_code:
            cmd.append('--confirm_run_unsafe_code')
        if limit is not None:
            cmd += ['--limit', str(limit)]
        print(f'\n===== {variant}: {task} =====', flush=True)
        run_live(cmd, env={
            'HF_ALLOW_CODE_EVAL': '1',
            'PYTHONUNBUFFERED': '1',
            # FlashInfer sampling JIT currently fails on Colab G4 Blackwell SM 12.x.
            # This only switches sampling to vLLM's native PyTorch/Triton path.
            'VLLM_USE_FLASHINFER_SAMPLER': '0',
        })
    print('Saved benchmark results:', result_root)
    return result_root

In [ ]:
# 1) Smoke test evaluation: chạy ít mẫu để kiểm tra checkpoint và dependencies.
evaluate_benchmarks('cst', limit=20, tasks=[BENCHMARKS[0]])

# 2) Full paper-style evaluation: chỉ chạy sau khi smoke test pass.
# evaluate_benchmarks('cst', limit=None)
# Nếu GSM8K đã xong nhưng MATH từng thiếu dependency, tiếp tục từ task thứ hai:
# evaluate_benchmarks('cst', limit=None, tasks=BENCHMARKS[1:])

# So sánh công bằng: chạy cùng danh sách task cho các checkpoint khác.
# evaluate_benchmarks('none', limit=None)
# evaluate_benchmarks('nuno', limit=None)
# evaluate_benchmarks('rpt', limit=None)